## 1. Титульный лист

# Инженерия признаков: прогнозирование покупки на сайте электронной коммерции

**Дисциплина:** Машинное обучение и искусственный интеллект  
**Модуль 4:** Инженерия признаков и работа с нестандартными данными  
**Датасет:** Online Shoppers Purchasing Intention  
**Выполнил:** Черепанов Вячеслав [отчество]  
**Группа:** [указать группу]  
**Преподаватель:** [ФИО преподавателя]  
**Репозиторий:** https://github.com/CherepanovVyacheslav/ml-course-cherepanov  
**Путь к работе:** `module-4-features/notebook.ipynb`  
**Дата:** 28.10.2026

Работа посвящена исследованию поведения покупателей и сравнению ансамблевых алгоритмов с акцентом на создание новых признаков.

## 2. Введение

**Постановка задачи.** По характеристикам пользовательской сессии необходимо предсказать, завершится ли она покупкой (`Revenue=1`). Это задача бинарной классификации: в отличие от регрессии, результатом служит принадлежность к одному из двух классов, а не непрерывное число.

Инженерия признаков — создание более информативных показателей из исходных данных. Например, совокупное время на сайте или средняя длительность просмотра страницы могут характеризовать вовлечённость точнее, чем отдельные счётчики.

**Практическая ценность.** Оценка вероятности покупки позволяет анализировать пути пользователей, расставлять приоритеты в улучшении сайта и оценивать аудитории рекламных кампаний. При этом важен момент получения признаков: показатели, измеряемые в конце сессии, нельзя использовать как будто они известны заранее.

В выборке около 15,5% положительных наблюдений, поэтому одной Accuracy недостаточно: алгоритм, всегда предсказывающий отсутствие покупки, уже получил бы около 84,5%. Основная метрика сравнения — **ROC-AUC** (способность ранжировать сессии), дополнительно рассматриваются **Precision, Recall и F1** для класса покупателей.

Цели работы: создать не менее пяти содержательных признаков; обучить и сравнить четыре ансамблевых алгоритма; подобрать параметры LightGBM и объяснить предсказания с помощью SHAP. Отдельно проверяется влияние дополнительных признаков на качество.

## 3. Теоретическая часть

**Feature Engineering.** Исходные данные не всегда отражают поведение клиента в форме, удобной модели. Например, количество страниц само по себе не говорит, как распределялось внимание пользователя. Поэтому из счётчиков и длительностей можно составить отношения, средние значения и взаимодействия. Новые признаки нужно проверять на полезность: усложнение не гарантирует улучшения. Отдельно необходимо следить за утечкой целевой переменной и доступностью признака в момент прогнозирования.

**Категориальные признаки.** `Label Encoding` заменяет категории целыми числами; для номинальных категорий это может создать ложное представление о порядке. `One-Hot Encoding` создаёт отдельный индикатор каждой категории и не навязывает фиктивное расстояние между ними. Для `VisitorType` и `Month` в работе используется One-Hot, а числовые коды браузера и региона также трактуются как категории.

**Временные признаки.** Месяц позволяет учесть сезонность, а бинарный выходной — различия между рабочими и свободными днями. В исходном наборе есть `Month` и `Weekend`, но нет календарной даты или времени отдельного события. Поэтому невозможно корректно рассчитать день недели и лаги по клиентам; не следует придумывать такие сведения.

**Поведенческие признаки.** Сумма счётчиков `TotalPages` описывает общий объём навигации, а `TotalDuration` — время, проведённое на сайте. Средние показатели на страницу снижают зависимость результата от общей длины сессии. Отношение времени на товарных страницах к общей длительности показывает распределение интереса. Деление выполняется с небольшим положительным знаменателем либо с обработкой нулевых значений.

**Дисбаланс классов.** При доле покупок примерно 15,5% модель может предпочитать отрицательный класс. Взвешивание классов увеличивает штраф за ошибки на редком классе, а SMOTE создаёт синтетические объекты только на обучающей части. Но увеличение Recall может ухудшить Precision, поэтому необходимо анализировать обе метрики. В основных моделях применяются веса классов, без искусственного изменения тестовой выборки.

**SHAP.** Метод опирается на значения Шепли и оценивает вклад отдельного признака в выход модели относительно базового уровня. Summary plot показывает общую картину, bar plot — среднюю абсолютную важность, waterfall plot — объяснение одной сессии. Dependence plot иллюстрирует связь значения признака с его вкладом. SHAP показывает объяснение модели, а не доказательство причинно-следственной связи.

**Ансамбли:** для усреднения базовых моделей (bagging) и аддитивной композиции (boosting) используют выражения

$$H_{bag}(x)=\frac{1}{M}\sum_{m=1}^{M}h_m(x),\qquad H_{boost}(x)=\sum_{m=1}^{M}\alpha_m h_m(x).$$

| Модель | Метод | Особенность | Ограничение |
|---|---|---|---|
| Random Forest | Bagging | Усреднение множества деревьев | Более громоздкий ансамбль |
| XGBoost | Boosting | Регуляризация и оптимизация дерева | Чувствителен к настройкам |
| LightGBM | Boosting | Быстрый рост деревьев по листьям | Может переобучаться на маленьких данных |
| CatBoost | Boosting | Хорошо работает с категориями | Обучение иногда занимает больше времени |

## 4. Описание данных

Основной источник — [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/468/online+shoppers+purchasing+intention+dataset), набор **Online Shoppers Purchasing Intention**. В исходном файле 12 330 сессий и 18 столбцов (17 признаков и `Revenue`). По описанию источника, 10 422 сессии завершились без покупки, 1 908 — с покупкой. *Уточнение:* технические коды `OperatingSystems`, `Browser`, `Region`, `TrafficType` хранятся как числа, но содержательно это категории.

| Группа | Столбцы | Смысл |
|---|---|---|
| Посещения | Administrative, Informational, ProductRelated | Количество просмотренных страниц |
| Длительность | Administrative_Duration, Informational_Duration, ProductRelated_Duration | Время просмотра |
| Метрики поведения | BounceRates, ExitRates, PageValues | Поведение в рамках сессии |
| Календарь | Month, SpecialDay, Weekend | Месяц, близость к празднику и выходной |
| Технические | OperatingSystems, Browser, Region, TrafficType | Сведения об окружении и канале |
| Посетитель | VisitorType | Новый или возвращающийся |
| Целевая | Revenue | Факт покупки |

В описании источника пропусков не заявлено, но перед обработкой это будет проверено программно. Файл `data/dataset_info.md` содержит сведения об источнике и признаках.

In [ ]:
# Загружаем данные из папки модуля; при отсутствии CSV запускаем загрузчик.
from pathlib import Path
import subprocess, sys, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Код рассчитан на запуск из module-4-features/.
DATA_PATH = Path('data/online_shoppers_intention.csv')
if not DATA_PATH.exists():
    subprocess.run([sys.executable, 'download_dataset.py'], check=True)

# Читаем исходный CSV; проверяем форму и распределение классов.
df = pd.read_csv(DATA_PATH)
print('Форма исходных данных:', df.shape)
print(df.head())
print('Типы:', df.dtypes.to_string())
print('Пропуски:', df.isna().sum().sum())
print('Revenue:', df['Revenue'].value_counts(dropna=False).to_dict())
assert df.shape == (12330, 18), 'Получен неожиданный размер датасета'
assert set(df['Revenue'].astype(str).str.lower()) == {'false', 'true'}

По данным документации ожидается заметный дисбаланс классов (84,5% против 15,5%). Поэтому в обучении обязательно используется `stratify=y`: доля покупок сохранится в обеих частях выборки. Пропуски и типы данных проверяются отдельно, даже если источник сообщает об их отсутствии.

## 5. Подготовка данных и EDA

Сначала проверим распределение целевой переменной, длительности просмотра товаров, стоимость страниц и связи числовых признаков. Для плохо читаемых графиков длительностей визуальный диапазон ограничим 99-м процентилем, **не удаляя** соответствующие сессии из данных.

In [ ]:
# Первичная диагностика количественных переменных.
print(df.info())
display(df.describe(include='all').T)
print('Доля покупок:', round(df['Revenue'].astype(int).mean() * 100, 2), '%')
print('Пропуски по столбцам:', df.isnull().sum().to_dict())

# Создаём шесть графиков и сохраняем их в папку examples/.
fig, ax = plt.subplots(2, 3, figsize=(18, 11))
sns.countplot(data=df, x='Revenue', ax=ax[0, 0], color='steelblue')
ax[0, 0].set(title='Покупка в сессии', xlabel='Revenue', ylabel='Число сессий')

# Ограничение применяется только при отображении длительностей.
visible = df[df['ProductRelated_Duration'] <= df['ProductRelated_Duration'].quantile(0.99)]
sns.histplot(data=visible, x='ProductRelated_Duration', hue='Revenue', bins=40, ax=ax[0, 1])
ax[0, 1].set(title='Время на товарных страницах', xlabel='Секунды', ylabel='Число сессий')

# Не выводим экстремальные точки boxplot, чтобы видеть центральную часть.
sns.boxplot(data=df, x='Revenue', y='PageValues', showfliers=False, ax=ax[0, 2], color='steelblue')
ax[0, 2].set(title='PageValues по классам', xlabel='Покупка', ylabel='PageValues')

# Анализируем корреляции только числовых признаков.
num = df.select_dtypes(include='number')
sns.heatmap(num.corr(), cmap='coolwarm', center=0, ax=ax[1, 0])
ax[1, 0].set_title('Корреляции числовых признаков')

# Сопоставляем долю покупок по типу посетителя.
sns.barplot(data=df, x='VisitorType', y='Revenue', ax=ax[1, 1], errorbar=None, color='steelblue')
ax[1, 1].set(title='Конверсия по типу посетителя', xlabel='Тип посетителя', ylabel='Доля покупок')
ax[1, 1].tick_params(axis='x', rotation=15)

# Сопоставляем долю покупок по календарным месяцам.
months = ['Feb', 'Mar', 'May', 'June', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
sns.barplot(data=df, x='Month', y='Revenue', order=months, ax=ax[1, 2], errorbar=None, color='steelblue')
ax[1, 2].set(title='Конверсия по месяцам', xlabel='Месяц', ylabel='Доля покупок')
fig.tight_layout()
Path('examples').mkdir(exist_ok=True)
fig.savefig('examples/eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()

# Отдельно визуализируем число пропусков в каждом столбце.
fig, ax = plt.subplots(figsize=(10, 4))
df.isna().sum().sort_values(ascending=False).plot.bar(ax=ax)
ax.set(title='Проверка пропусков по столбцам', xlabel='Признак', ylabel='Число пропусков')
fig.tight_layout()
fig.savefig('examples/missing_values.png', dpi=150, bbox_inches='tight')
plt.show()

**Наблюдения и обработка.** Распределение покупок неравномерно, а длительности просмотра и `PageValues` имеют длинный правый хвост. На boxplot и гистограмме видно, что характерные значения различаются у двух классов; это связь, но не причинное объяснение. Посетители разных типов и месяцы показывают неодинаковую конверсию. Пропуски в оригинальном CSV отсутствуют; всё равно предусмотрим заполнение возможных пустых числовых значений **медианами обучающей выборки**. Это надёжнее, чем вычислять медиану по всей таблице до разделения.

Создаём признаки, которые описывают *общее количество действий*, *интенсивность*, *относительную товарную активность* и *условия сессии*. Из-за отсутствия полной временной шкалы не создаём фиктивные лаги или скользящие окна.

In [ ]:
# Маппинг месяцев нужен для обычного числового календарного признака.
MONTH_MAP = {'Jan':1, 'Feb':2, 'Mar':3, 'Apr':4, 'May':5, 'June':6,
             'Jul':7, 'Aug':8, 'Sep':9, 'Oct':10, 'Nov':11, 'Dec':12}
CAT_COLS = ['Month', 'VisitorType', 'OperatingSystems', 'Browser', 'Region', 'TrafficType']

# Функция не использует Revenue: признаки можно построить для новой сессии.
def make_features(raw):
    f = raw.drop(columns=['Revenue'], errors='ignore').copy()
    f['TotalPages'] = f['Administrative'] + f['Informational'] + f['ProductRelated']
    f['TotalDuration'] = (f['Administrative_Duration'] + f['Informational_Duration']
                          + f['ProductRelated_Duration'])
    f['AvgPageValue'] = f['PageValues'] / (f['TotalPages'] + 1)
    f['BounceExitRatio'] = f['BounceRates'] / (f['ExitRates'] + 0.001)
    f['AvgDurationPerPage'] = f['TotalDuration'] / (f['TotalPages'] + 1)
    f['ProductTimeShare'] = f['ProductRelated_Duration'] / (f['TotalDuration'] + 1)
    f['ProductPageShare'] = f['ProductRelated'] / (f['TotalPages'] + 1)
    f['IsNewVisitor'] = f['VisitorType'].eq('New_Visitor').astype(int)
    f['IsWeekend'] = f['Weekend'].astype(str).str.lower().isin(['true', '1']).astype(int)
    f['MonthNum'] = f['Month'].map(MONTH_MAP).fillna(0).astype(int)
    f['IsHighTraffic'] = (pd.to_numeric(f['TrafficType'], errors='coerce') > 10).astype(int)
    f['HighEngagement'] = ((f['TotalPages'] > 30) & (f['TotalDuration'] > 1000)).astype(int)
    f['PageValueDurationInteraction'] = f['PageValues'] * np.log1p(f['TotalDuration'].clip(lower=0))
    f = f.drop(columns=['Weekend'])

    # Категориальные коды преобразуем в строки, затем — в независимые индикаторы.
    for col in CAT_COLS:
        f[col] = f[col].astype(str).fillna('Unknown')
    f = pd.get_dummies(f, columns=CAT_COLS, dtype=int)
    return f.replace([np.inf, -np.inf], np.nan)

# Деление выполняем ДО вычисления статистик заполнения и масштабирования.
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
raw_y = df['Revenue'].astype(str).str.lower().map({'true':1, 'false':0})
raw_X = df.drop(columns='Revenue')
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    raw_X, raw_y, test_size=0.2, random_state=42, stratify=raw_y)

# Список признаков и статистики заполнения определяются только по train.
X_train = make_features(X_train_raw)
feature_cols = X_train.columns.to_list()
X_test = make_features(X_test_raw).reindex(columns=feature_cols, fill_value=0)
train_medians = X_train.median(numeric_only=True).fillna(0)
X_train = X_train.fillna(train_medians).astype(float)
X_test = X_test.fillna(train_medians).astype(float)

# Сохраняем имена столбцов после кодирования в том же порядке.
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_cols, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=feature_cols, index=X_test.index)
print('Обучающая и тестовая части:', X_train_scaled.shape, X_test_scaled.shape)
print('Новых/производных признаков: 13; признаков после One-Hot:', len(feature_cols))
print('Доли Revenue=1:', round(y_train.mean(), 4), round(y_test.mean(), 4))

**Обоснование признаков.** `TotalPages` и `TotalDuration` отражают масштаб активности; `AvgPageValue` и `AvgDurationPerPage` делают значения сопоставимыми между короткими и длинными сессиями. `BounceExitRatio` связывает две метрики ухода со страниц, а `ProductTimeShare` и `ProductPageShare` показывают долю интереса к товарам. `IsNewVisitor`, `IsWeekend`, `MonthNum`, `IsHighTraffic` добавляют информацию о контексте визита. `HighEngagement` описывает интенсивную сессию, а `PageValueDurationInteraction` проверяет, усиливается ли влияние ценности страницы при долгом просмотре. Пороговые значения здесь исследовательские, а не универсальные бизнес-правила. Масштабирование деревьям обычно не нужно, но единое представление признаков упрощает сохранение и повторную загрузку.

## 6. Построение и обучение моделей

Используются две группы методов: усреднение деревьев (Random Forest) и градиентный бустинг (XGBoost, LightGBM, CatBoost). Фиксируем `random_state=42` и измеряем время обучения. Положительный класс встречается реже, поэтому в алгоритмах используются веса классов. Масштабирование выполнено одинаково для всех моделей; для деревьев это не является обязательным требованием.

In [ ]:
# Подключаем ансамблевые модели и измеряем фактическое время обучения.
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Вес положительного класса вычисляется строго на обучающей части.
pos_weight = (y_train.eq(0).sum() / y_train.eq(1).sum())
models = {
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=10,
                                           class_weight='balanced', n_jobs=-1, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                             subsample=0.9, colsample_bytree=0.9,
                             scale_pos_weight=pos_weight, eval_metric='logloss',
                             n_jobs=2, random_state=42),
    'LightGBM': LGBMClassifier(n_estimators=200, num_leaves=31, max_depth=7,
                               learning_rate=0.05, class_weight='balanced',
                               n_jobs=2, random_state=42, verbosity=-1),
    'CatBoost': CatBoostClassifier(iterations=200, depth=6, learning_rate=0.1,
                                   auto_class_weights='Balanced', thread_count=2,
                                   random_seed=42, verbose=False)
}
training_time = {}
for name, model in models.items():
    t0 = time.perf_counter()
    model.fit(X_train_scaled, y_train)
    training_time[name] = time.perf_counter() - t0
    print(f'{name}: {training_time[name]:.2f} сек.')

Random Forest служит устойчивым базовым ориентиром. XGBoost, LightGBM и CatBoost используют последовательное уточнение ансамбля и позволяют сравнить семейства бустинга. Более высокая сложность модели не гарантирует лучший результат: это проверяем на отложенных данных.

## 7. Оценка качества

Для всех четырёх алгоритмов рассчитаем Accuracy, Precision, Recall, F1 и ROC-AUC, а также построим ROC-кривые и матрицу ошибок. Показатели вычисляем на одной и той же тестовой части, не участвовавшей в обучении.

In [ ]:
# ROC-AUC вычисляется по вероятностям, остальные метрики — по меткам классов.
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, classification_report)
def measure(model, X, y, name):
    pred = model.predict(X).astype(int).reshape(-1)
    prob = model.predict_proba(X)[:, 1]
    return {'Model': name, 'Accuracy': accuracy_score(y, pred),
            'Precision': precision_score(y, pred, zero_division=0),
            'Recall': recall_score(y, pred, zero_division=0),
            'F1': f1_score(y, pred, zero_division=0),
            'ROC-AUC': roc_auc_score(y, prob)}

# Выводим сводную таблицу без pandas.style (не зависит от Jinja2).
results_df = pd.DataFrame([measure(model, X_test_scaled, y_test, name)
                           for name, model in models.items()]).sort_values('ROC-AUC', ascending=False)
print(results_df.round(4).to_string(index=False))

# ROC-кривые сохраняем для сравнения всех четырёх моделей.
fig, ax = plt.subplots(figsize=(9, 6))
for name, model in models.items():
    prob = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, prob)
    ax.plot(fpr, tpr, label=f'{name}: {roc_auc_score(y_test, prob):.3f}')
ax.plot([0, 1], [0, 1], '--', color='grey', label='Случайная модель')
ax.set(xlabel='FPR', ylabel='TPR', title='ROC-кривые: ансамблевые модели')
ax.grid(alpha=.25)
ax.legend()
fig.savefig('examples/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# Матрица ошибок лучшего исходного алгоритма по ROC-AUC.
baseline_name = results_df.iloc[0]['Model']
base_best = models[baseline_name]
cm = confusion_matrix(y_test, base_best.predict(X_test_scaled))
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Без покупки','Покупка'], yticklabels=['Без покупки','Покупка'])
ax.set(xlabel='Предсказано', ylabel='Истинно', title=f'Матрица ошибок: {baseline_name}')
fig.tight_layout()
fig.savefig('examples/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Лучшая исходная модель по ROC-AUC:', baseline_name)
print(classification_report(y_test, base_best.predict(X_test_scaled), zero_division=0))

**Как читать результаты.** ROC-AUC показывает качество ранжирования без выбора фиксированного порога. Precision отвечает на вопрос, какова доля реальных покупателей среди сессий, отмеченных моделью как покупка; Recall показывает, какую долю покупок удалось обнаружить. При взвешивании редкого класса Recall нередко растёт ценой Precision. Таблица выше содержит действительные значения, рассчитанные при запуске, а не цифры из примера в методичке.

## 8. Интерпретация и бизнес-инсайты

Сравним важность признаков по Random Forest и LightGBM. У деревьев она отражает вклад переменных в разделения, но не направление влияния и не причинность. Для направления эффекта ниже используем SHAP.

In [ ]:
# Формируем две таблицы важности на одних и тех же признаках.
imp_rf = pd.Series(models['Random Forest'].feature_importances_, index=feature_cols)
imp_lgbm = pd.Series(models['LightGBM'].feature_importances_, index=feature_cols)
important = pd.DataFrame({'Random Forest': imp_rf, 'LightGBM': imp_lgbm})
print(important.sort_values('LightGBM', ascending=False).head(15).round(3).to_string())

# Топ-15 для исходной LightGBM; сохраним график в examples/.
fig, ax = plt.subplots(figsize=(9, 6))
imp_lgbm.nlargest(15).sort_values().plot.barh(ax=ax)
ax.set(xlabel='Важность (частота использования в разделениях)',
       title='15 значимых признаков LightGBM')
fig.tight_layout()
fig.savefig('examples/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

# Выводим наблюдаемые конверсии в группах: это описание, не причинная оценка.
print('Конверсия по типам посетителей, %:')
print((df.groupby('VisitorType')['Revenue'].mean() * 100).round(2).to_string())
print('Конверсия по выходным, %:')
print((df.groupby('Weekend')['Revenue'].mean() * 100).round(2).to_string())

# Абляция: оценим влияние созданных признаков на ROC-AUC той же LightGBM.
engineered_names = ['TotalPages', 'TotalDuration', 'AvgPageValue',
                    'BounceExitRatio', 'AvgDurationPerPage', 'ProductTimeShare',
                    'ProductPageShare', 'IsNewVisitor', 'IsWeekend', 'MonthNum',
                    'IsHighTraffic', 'HighEngagement', 'PageValueDurationInteraction']
original_columns = [col for col in feature_cols if col not in engineered_names]
from sklearn.base import clone
no_fe_model = clone(models['LightGBM'])
no_fe_model.fit(X_train_scaled[original_columns], y_train)
auc_original = roc_auc_score(y_test, no_fe_model.predict_proba(X_test_scaled[original_columns])[:, 1])
auc_engineered = roc_auc_score(y_test, models['LightGBM'].predict_proba(X_test_scaled)[:, 1])
print(f'LightGBM без новых признаков: ROC-AUC={auc_original:.4f}')
print(f'LightGBM с новыми признаками: ROC-AUC={auc_engineered:.4f}')
print(f'Разница: {auc_engineered-auc_original:+.4f} (тестовая выборка)')

**Бизнес-смысл.** Важность `PageValues`, показателей отказов и времени просмотра помогает понять, какие сигналы модель использует для ранжирования сессий. Суммарные и относительные метрики удобны для мониторинга качества навигации и вовлечённости. При интерпретации важно помнить: если `PageValues` известен только к концу визита, полученная модель подходит для анализа завершённых сессий, но не для раннего прогноза в начале посещения. Для такого сценария нужно отдельно собирать события по времени.

## 9. Тюнинг гиперпараметров (Optuna)

Для LightGBM проведём 30 испытаний Optuna. Целевая метрика — средний ROC-AUC по пяти стратифицированным фолдам **только на обучающей части**. Это позволяет не подбирать параметры под тестовую выборку. После завершения поиска обучим итоговый LightGBM на всём train и один раз оценим его на test.

In [ ]:
# Оптимизация параметров LightGBM с воспроизводимым семенем.
import optuna
from sklearn.model_selection import StratifiedKFold, cross_val_score
optuna.logging.set_verbosity(optuna.logging.WARNING)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def objective(trial):
    pars = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 400, step=50),
        'num_leaves': trial.suggest_int('num_leaves', 10, 80, step=5),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.65, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.65, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 60),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 5.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 5.0, log=True),
    }
    m = LGBMClassifier(**pars, class_weight='balanced',
                       random_state=42, n_jobs=2, verbosity=-1)
    scores = cross_val_score(m, X_train_scaled, y_train, scoring='roc_auc', cv=cv, n_jobs=1)
    return float(scores.mean())

# TPE — последовательный вероятностный поиск. Минимум из методички: 20 испытаний.
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
t0 = time.perf_counter()
study.optimize(objective, n_trials=30, show_progress_bar=False)
tune_time = time.perf_counter() - t0
print('Оптимизация, время:', round(tune_time, 2), 'с')
print('Лучший CV ROC-AUC:', round(study.best_value, 5))
print('Параметры:', study.best_params)

# Находим итоговую модель и проверяем её на нетронутом тесте.
best_lgbm = LGBMClassifier(**study.best_params, class_weight='balanced',
                            random_state=42, n_jobs=2, verbosity=-1)
t0 = time.perf_counter()
best_lgbm.fit(X_train_scaled, y_train)
training_time['LightGBM (Optuna)'] = time.perf_counter() - t0
optuna_result = measure(best_lgbm, X_test_scaled, y_test, 'LightGBM (Optuna)')
base_lgbm_result = measure(models['LightGBM'], X_test_scaled, y_test, 'LightGBM (baseline)')
print(pd.DataFrame([base_lgbm_result, optuna_result]).round(4).to_string(index=False))
print('Изменение ROC-AUC на тесте:', round(optuna_result['ROC-AUC'] - base_lgbm_result['ROC-AUC'], 5))

Здесь особенно важно различать два числа: лучший **кросс-валидационный** ROC-AUC относится к поиску параметров на train; ROC-AUC в таблице сравнения baseline/Optuna — к отложенному test. Улучшение после настройки не гарантировано: если разница отрицательная, это честный результат проверки, а не повод повторно подбирать параметры по тесту.

## 10. Интерпретация через SHAP

Интерпретируем оптимизированную LightGBM. Для скорости визуализации возьмём первые 300 объектов из отложенной выборки, сохранив исходный порядок. Расчёт производится для выходной оценки модели; её единицы (часто log-odds) не следует без преобразования считать изменениями вероятности в процентных пунктах.

In [ ]:
# SHAP: учитываем различия форматов результата у разных версий библиотеки.
import shap
shap_input = X_test_scaled.iloc[:300]
explainer = shap.TreeExplainer(best_lgbm)
exp = explainer(shap_input)
if exp.values.ndim == 3:
    # Для бинарной классификации некоторые версии возвращают оба класса.
    sv = exp.values[:, :, 1]
    bv = exp.base_values[:, 1] if np.asarray(exp.base_values).ndim > 1 else exp.base_values
else:
    sv = exp.values
    bv = exp.base_values

# Summary plot показывает вклад всех признаков для нескольких сессий.
shap.summary_plot(sv, shap_input, show=False, max_display=18)
plt.gcf().savefig('examples/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

# Bar plot показывает среднюю абсолютную важность.
shap.summary_plot(sv, shap_input, plot_type='bar', show=False, max_display=18)
plt.gcf().savefig('examples/shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

# Объясняем одну сессию с самой высокой вероятностью покупки в выбранном фрагменте.
pred_sub = best_lgbm.predict_proba(shap_input)[:, 1]
idx = int(np.argmax(pred_sub))
base_scalar = np.asarray(bv).reshape(-1)[idx] if np.asarray(bv).ndim > 0 else float(bv)
instance = shap.Explanation(values=sv[idx], base_values=base_scalar,
                            data=shap_input.iloc[idx].to_numpy(),
                            feature_names=feature_cols)
shap.plots.waterfall(instance, max_display=16, show=False)
plt.gcf().savefig('examples/shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()
print('Вероятность покупки для объясняемой сессии:', round(float(pred_sub[idx]), 4))

# График зависимости для наиболее влиятельного признака по абсолютному SHAP.
top_feat = feature_cols[int(np.abs(sv).mean(axis=0).argmax())]
shap.dependence_plot(top_feat, sv, shap_input, interaction_index=None, show=False)
plt.gcf().savefig('examples/shap_dependence.png', dpi=150, bbox_inches='tight')
plt.show()
print('Наиболее значимый по SHAP признак:', top_feat)

Summary plot позволяет увидеть, какие значения признаков сдвигают выход модели сильнее остальных, а waterfall — почему конкретная сессия получила высокую или низкую оценку. Особенно осторожно следует трактовать искусственно сконструированные отношения и взаимодействия: похожие признаки коррелируют и могут разделять вклад между собой. SHAP объясняет **предсказание модели**, а не подтверждает, что изменение показателя само по себе вызовет покупку.

## 11. Сохранение модели и метаданных

Сохраняем четыре исходные модели, LightGBM после Optuna, масштабировщик, порядок признаков, статистики заполнения и метрики. Сохранять только `.pkl` недостаточно: новая сессия должна пройти точно такую же предобработку.

In [ ]:
# Сохраняем все модели и преобразователь с помощью joblib.
import joblib
MODEL_DIR = Path('models')
MODEL_DIR.mkdir(exist_ok=True)
file_names = {'Random Forest': 'rf_model.pkl', 'XGBoost': 'xgb_model.pkl',
              'LightGBM': 'lgbm_model.pkl', 'CatBoost': 'cat_model.pkl'}
for name, fn in file_names.items():
    joblib.dump(models[name], MODEL_DIR / fn)
joblib.dump(best_lgbm, MODEL_DIR / 'best_model.pkl')
joblib.dump(scaler, MODEL_DIR / 'scaler.pkl')

# Метаданные позволяют воспроизвести предобработку новых объектов.
(MODEL_DIR / 'feature_cols.json').write_text(
    json.dumps(feature_cols, ensure_ascii=False, indent=2), encoding='utf-8')
(MODEL_DIR / 'medians.json').write_text(
    json.dumps({k:float(v) for k,v in train_medians.items()}, indent=2), encoding='utf-8')
metrics_out = {
    'baseline': {r['Model']: {k:float(v) for k,v in r.items() if k != 'Model'}
                 for r in results_df.to_dict('records')},
    'optuna': {k:float(v) for k,v in optuna_result.items() if k != 'Model'},
    'best_model': 'LightGBM (Optuna)', 'best_cv_roc_auc': float(study.best_value),
    'best_params': study.best_params,
    'training_time_sec': {k:float(v) for k,v in training_time.items()},
    'split': {'test_size':0.2, 'random_state':42, 'stratified':True},
    'feature_count':len(feature_cols)
}
(MODEL_DIR / 'metrics.json').write_text(
    json.dumps(metrics_out, ensure_ascii=False, indent=2), encoding='utf-8')
print('Файлы:', sorted(p.name for p in MODEL_DIR.iterdir()))

Порядок столбцов в `feature_cols.json` фиксирован. При инференсе отсутствующие после One-Hot категории добавляются нулевыми столбцами. `scaler.pkl` и `medians.json` обучены только на train, поэтому повторное преобразование не использует статистику тестовой выборки.

## 12. Загрузка модели из репозитория

После публикации файлов в ветке `main` модель можно скачать через адрес `raw.githubusercontent.com`. Сначала проверим локальную загрузку, затем, если файлы уже опубликованы, сравним три предсказания загруженного удалённо экземпляра с исходным. Файлы `joblib` следует десериализовать **только из собственного доверенного репозитория**.

In [ ]:
# Согласованная предобработка нового объекта — та же, что при обучении.
def predict_purchase(raw_records, model, fitted_scaler, ordered_columns, medians):
    f = make_features(pd.DataFrame(raw_records)).reindex(columns=ordered_columns, fill_value=0)
    f = f.fillna(pd.Series(medians)).astype(float)
    transformed = pd.DataFrame(fitted_scaler.transform(f), columns=ordered_columns)
    return model.predict_proba(transformed)[:, 1]

# Локальная проверка сохранённых файлов и трёх прогнозов.
loaded_local = joblib.load(MODEL_DIR / 'best_model.pkl')
loaded_scaler = joblib.load(MODEL_DIR / 'scaler.pkl')
loaded_cols = json.loads((MODEL_DIR / 'feature_cols.json').read_text(encoding='utf-8'))
loaded_medians = json.loads((MODEL_DIR / 'medians.json').read_text(encoding='utf-8'))
raw_3 = X_test_raw.iloc[:3].to_dict('records')
original_proba = best_lgbm.predict_proba(X_test_scaled.iloc[:3])[:, 1]
local_proba = predict_purchase(raw_3, loaded_local, loaded_scaler, loaded_cols, loaded_medians)
np.testing.assert_allclose(original_proba, local_proba, rtol=1e-7, atol=1e-8)
print('Локальная загрузка: прогнозы совпадают:', np.round(local_proba, 5))

# Проверяем raw-ссылку и не выдаём локальный файл за файл GitHub.
import requests
from io import BytesIO
GITHUB_RAW_BASE = ('https://raw.githubusercontent.com/CherepanovVyacheslav/'
                   'ml-course-cherepanov/main/module-4-features')
try:
    url = f'{GITHUB_RAW_BASE}/models/best_model.pkl'
    response = requests.get(url, timeout=12)
    response.raise_for_status()
    remote_model = joblib.load(BytesIO(response.content))
    remote_scaler_response = requests.get(f'{GITHUB_RAW_BASE}/models/scaler.pkl', timeout=12)
    remote_scaler_response.raise_for_status()
    remote_scaler = joblib.load(BytesIO(remote_scaler_response.content))
    remote_medians_response = requests.get(f'{GITHUB_RAW_BASE}/models/medians.json', timeout=12)
    remote_medians_response.raise_for_status()
    remote_medians = remote_medians_response.json()
    remote_cols_response = requests.get(f'{GITHUB_RAW_BASE}/models/feature_cols.json', timeout=12)
    remote_cols_response.raise_for_status()
    remote_cols = remote_cols_response.json()
    assert remote_cols == loaded_cols
    remote_proba = predict_purchase(raw_3, remote_model, remote_scaler, remote_cols, remote_medians)
    np.testing.assert_allclose(remote_proba, original_proba, rtol=1e-7, atol=1e-8)
    print('GitHub raw: три прогноза совпадают:', np.round(remote_proba, 5))
except (requests.RequestException, ValueError, AssertionError) as exc:
    print('Проверка GitHub ещё не пройдена:', str(exc)[:180])
    print('Опубликуйте models/ в ветке main и повторите эту ячейку.')

Проверка сначала сравнивает локально загруженные модель и масштабировщик с объектами в памяти — без переобучения. Удалённая проверка считается выполненной только при доступной raw-ссылке и совпадении вероятностей. HTTP 404 означает, что модель ещё не размещена по указанному пути или используется неправильная ветка.

## 13. Выводы

**Что получилось.** Построены 13 производных показателей активности и вовлечённости, выполнено кодирование категориальных полей, сравнены четыре ансамблевых алгоритма. Реальные значения Accuracy, F1 и ROC-AUC выводятся в разделе 7 после запуска. Параметры LightGBM подбираются в 30 испытаниях Optuna с пятикратной проверкой на train. Основные факторы и отдельные решения модели исследуются с помощью графиков SHAP.

**Основные трудности.** На результат заметно влияет дисбаланс классов. Некоторые признаки сильно коррелируют: длительность сессии, число страниц и производные отношения описывают близкие стороны поведения. Кроме того, показатели `PageValues` и итоговая длительность могут стать известны только в конце сессии, поэтому необходима осторожность при интерпретации результатов как прогноза «в реальном времени». Подбор гиперпараметров увеличивает время выполнения, а различные версии SHAP могут возвращать разные формы массивов.

**Возможные улучшения.** (1) Сравнить результаты с исключённым `PageValues`, чтобы проверить сценарий раннего прогноза. (2) Проверить взаимодействие `PageValues × TotalDuration` отдельной абляцией. (3) Изучить Target Encoding категорий строго внутри фолдов. (4) Применить SMOTE только к тренировочным фолдам и сравнить F1/ROC-AUC. (5) Откалибровать вероятности и отдельно настроить порог по бизнес-стоимости ошибок. (6) Организовать Feature Store и контроль дрейфа признаков. (7) Получить временные события и построить честные лаговые признаки.

Объективная оценка работы должна опираться на полученные при запуске метрики; значения в методических примерах не используются как результат эксперимента.

## 14. Источники и приложение

1. Sakar C., Kastro Y. **Online Shoppers Purchasing Intention Dataset**. UCI Machine Learning Repository (2018). https://doi.org/10.24432/C5F88Q
2. Sakar C. O. et al. *Real-time prediction of online shoppers' purchasing intention using multilayer perceptron and LSTM recurrent neural networks*. Neural Computing and Applications (2019). https://doi.org/10.1007/s00521-018-3523-0
3. Breiman L. *Random Forests*. Machine Learning, 45, 5–32 (2001). https://doi.org/10.1023/A:1010933404324
4. Chen T., Guestrin C. *XGBoost: A Scalable Tree Boosting System*. KDD (2016). https://doi.org/10.1145/2939672.2939785
5. Ke G. et al. *LightGBM: A Highly Efficient Gradient Boosting Decision Tree*. NeurIPS (2017). https://papers.nips.cc/paper/6907-lightgbm-a-highly-efficient-gradient-boosting-decision-tree
6. Prokhorenkova L. et al. *CatBoost: unbiased boosting with categorical features*. NeurIPS (2018). https://arxiv.org/abs/1706.09516
7. Akiba T. et al. *Optuna: A Next-generation Hyperparameter Optimization Framework*. KDD (2019). https://arxiv.org/abs/1907.10902
8. Lundberg S. M., Lee S.-I. *A Unified Approach to Interpreting Model Predictions*. NeurIPS (2017). https://arxiv.org/abs/1705.07874

### Приложение: структура и запуск

Полный код размещён **в соответствующих ячейках выше** и снабжён комментариями. Воспроизводимое окружение описано в `requirements.txt`; для Windows включён `run_all_windows.bat`. Данные хранятся в `data/`, результаты — в `examples/`, обученные модели и метаданные — в `models/`.

Установка: `python -m pip install -r requirements.txt`  
Запуск: `jupyter notebook notebook.ipynb` (в папке `module-4-features`).

После публикации в GitHub необходимо открыть раздел 12 и подтвердить загрузку по raw-ссылке. Проект с модулями 1–4 хранится в одном репозитории `ml-course-cherepanov`.

In [ ]:
# Контроль наличия ключевых результатов и предписанного состава работы.
expected = ['rf_model.pkl', 'xgb_model.pkl', 'lgbm_model.pkl', 'cat_model.pkl',
            'best_model.pkl', 'scaler.pkl', 'feature_cols.json', 'metrics.json']
missing = [p for p in expected if not (MODEL_DIR / p).exists()]
print('Обязательных файлов моделей:', len(expected), '| отсутствуют:', missing)
assert not missing, 'Сначала выполните ячейки обучения и сохранения'
print('Графики:', sorted(p.name for p in Path('examples').glob('*.png')))